# LangChain: Evaluation (Groq Llama 3.1)

## Outline
* **Example Generation** - Create test data (manual + LLM-generated)
* **Manual Evaluation** - Debug mode to inspect chain internals
* **LLM-Assisted Evaluation** - Use LLM to grade responses
* **Evaluation Platform** - Track and visualize results

**Model Used:** Groq Llama-3.1-8b-instant  
**Key Concept:** Evaluating LLM applications is challenging because outputs are open-ended text, not exact matches.

**Why This Matters:** You need to know if your changes improve or degrade your application's performance.


In [ ]:
# Cell 2: Install Required Packages

!pip install langchain langchain-groq python-dotenv docarray sentence-transformers -q

In [ ]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


In [ ]:
# Cell 4: Create Q&A Application

from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain.document_loaders import CSVLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load data
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

print(f"✅ Loaded {len(data)} documents from {file}")


In [ ]:
# Cell 5: Create Vector Store Index

# Use HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create index
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings
).from_loaders([loader])

print("✅ Vector store index created")


In [ ]:
# Cell 6: Initialize LLM and QA Chain

# Initialize Groq LLM
llm_model = "llama-3.1-8b-instant"
llm = ChatGroq(temperature=0.0, model=llm_model, groq_api_key=groq_api_key)

# Create RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs={
        "document_separator": "<<<<>>>>>"
    }
)

print("✅ QA chain created successfully")


In [ ]:
# Cell 7: Inspect Sample Documents

# Look at some sample documents to understand the data
print("Document 10:")
print(data[10])
print("\n" + "="*60 + "\n")

print("Document 11:")
print(data[11])


## Coming Up With Test Data

We need examples to evaluate our QA system. There are two approaches:
1. **Manual** - Create examples by hand (time-consuming but precise)
2. **LLM-Generated** - Use an LLM to generate examples (fast but needs review)


In [ ]:
# Cell 8: Create Hard-Coded Examples

# Manually created question-answer pairs based on the documents above
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

print(f"✅ Created {len(examples)} manual examples")


In [ ]:
# Cell 9: LLM-Generated Examples

from langchain.evaluation.qa import QAGenerateChain

# Create a chain that generates Q&A pairs from documents
example_gen_chain = QAGenerateChain.from_llm(
    ChatGroq(model=llm_model, groq_api_key=groq_api_key)
)

print("✅ QA generation chain created")


In [ ]:
# Cell 10: Generate Examples from Documents

# Generate Q&A pairs from first 5 documents
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

print(f"✅ Generated {len(new_examples)} new examples")
print("\nFirst generated example:")
print(new_examples[0])
